In [3]:
import datetime, calendar, glob, os, requests, pickle

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.colors as mcolors

from matplotlib.ticker import FuncFormatter
from matplotlib import cm
from matplotlib_scalebar.scalebar import ScaleBar
from matplotlib.transforms import ScaledTranslation

path_to_validation = '/Users/Guille/Desktop/dynamic_update/validation/'
path_to_tables = '/Users/Guille/Desktop/dynamic_update/tables/'

In [4]:
def _load_pit(hyper_meta_, resource, method, type, interval, description, path):
    
    pit_ = pd.read_csv(path + f"/{resource}/{resource}_{method}_{type}-PIT_{interval}-{description}.csv")
    
    init = hyper_meta_.loc[hyper_meta_['time'] == interval, 'initialization'].values[0]
    pit_ = pit_.loc[pit_['initialization'] == init].reset_index(drop = True)

    return pit_.drop(columns = ['initialization', 'time']).to_numpy()

# solar asset
resource = 'solar'
method = 'fusion'
type = 'asset'
interval = 144
description = 1
pit_ = _load_pit(hyper_meta_, resource, method, type, interval, description, 
                 path = path_to_validation)

print(pit_.shape)

NameError: name 'hyper_meta_' is not defined

In [ ]:
def clean_fmt(x, pos):
    if abs(x - round(x)) < 1e-9:
        return str(int(round(x)))
    return f"{x:.2g}"

def _plot_PIT(_ax, u_, 
              bins=10,
              v=0.5,
              xlabel = 'PIT'):
    
    vmin= -v
    vmax= v
    
    KS_stat = _KS(u_)
    u_ = np.sort(u_)
    n = len(u_)

    # --- Histogram ---
    counts_, bins_, patches = _ax.hist(u_,
                                       bins=bins,
                                       range=(0.0, 1.0),
                                       density=True,
                                       lw=0.5,
                                       edgecolor='k',
                                       alpha=1)

    # deviation from perfect calibration
    deviation = counts_ - 1.
    #print(deviation.min(), deviation.max())
    #print(deviation)
    
    # --- Colormap centered at 0 ---
    norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

    cmap = cm.get_cmap("coolwarm")

    for patch, dev in zip(patches, deviation):
        #patch.set_facecolor(cmap(norm(dev)))
        patch.set_facecolor('lightgray')

    # --- Reference line ---
    _ax.axhline(1.0, 
                c = 'k',
                linestyle='--')

    _ax.set_xlim(0, 1)
    _ax.set_ylim(0, 3)

    #_ax.set_xlabel("u = F̂(y_obs)")
    _ax.set_xlabel(xlabel, size=14)
    _ax.set_ylabel("Density", size=14)

    _ax.xaxis.set_major_formatter(FuncFormatter(clean_fmt))

    _ax.tick_params(axis="both", labelsize=12)

    # --- Annotate KS statistic ---
    _ax.text(0.25, 0.95,
             f"$D_{{KS}}$ = {KS_stat:.3f}",
             transform=_ax.transAxes,
             verticalalignment='top')



_fig, _ax = plt.subplots(1, 1, 
                        figsize = (2.3, 2.25), 
                        layout  = "constrained")

horizon = 42
lead = 6
bins = 10

_plot_PIT(_ax, pit_[:, horizon:(horizon + lead)].flatten(), bins,
          xlabel = 'PIT (1h to 48h lead)')

plt.savefig(path_to_images + f"/PIT-zone-wind-{horizon}-{lead}.pdf", 
            bbox_inches="tight")

plt.show()